# **Fine-Tuning CharsiuG2P on the Wiktionary Training Data**

`ByT5_eval.ipynb` tests an existing Filipino neural G2P model, but that model learned from
different, machine-made labels for a different task (stress-aware sentences). This notebook asks the
controlled question: **is a neural model better than the WFST when both learn from the same data?**
It is the controlled neural candidate in `G2P_compare.ipynb`.

It fine-tunes CharsiuG2P, `charsiu/g2p_multilingual_byT5_small_100` (Zhu, Zhang & Jurgens, 2022),
the multilingual word-level ByT5 model that `filipino-byt5-g2p` also starts from, on the same
Wiktionary splits as the WFST model of `Wik_eval.ipynb`:

- The number of training epochs is chosen on the validation split (`dicts/cwik_val.dict`), after
  training on the training split (`dicts/cwik_train.dict`).
- The final model is trained on the training and validation pool (`dicts/cwik_trainval.dict`, 90%),
  the same data as the final WFST model, and tested once on the test split.
- It is cross-validated on the same 10 folds as the WFST model: retrained on nine folds and tested
  on the tenth.

CharsiuG2P was pretrained on WikiPron data in about 100 languages, including Tagalog, so it has
seen some of the test words before. Its Tagalog transcriptions are phonetic (`ʔɐbo`), not
phonemic like the references, so fine-tuning has to teach it this dictionary's conventions.

**Running it.** Fine-tuning needs an NVIDIA GPU; on a CPU it would take days. It doesn't need
Phonetisaurus, so it runs directly on Windows (outside the Docker container), in the same project
folder. Run `Wik_eval.ipynb` in the container first: it writes `notebook/dicts/`, which this notebook
reads. Then, in PowerShell at the project root, set up a separate GPU environment,
`.venv-windows`, with [`uv`](https://docs.astral.sh/uv/):

```powershell
uv venv .venv-windows --python 3.11
.venv-windows\Scripts\activate
uv pip install torch --index-url https://download.pytorch.org/whl/cu128
uv pip install transformers pandas numpy ipykernel
```

`cu128` is the CUDA 12.8 build of PyTorch; pick the build that matches your GPU driver on
[pytorch.org](https://pytorch.org/get-started/locally/). Open this notebook in VS Code, choose the
`.venv-windows` kernel and run all cells. The first cell stops with a message if PyTorch can't see the
GPU. The cross-validation trains ten more models, so it takes about ten times as long as the rest.
Its progress is saved after every fold, so a run that stops can be resumed.

Setting the environment variable `G2P_SMOKE_TEST=1` runs everything on a few words for one epoch,
to check the code on a CPU without a GPU. Its results mean nothing and are written to a temporary folder.

**The process at a glance**

1. **Load** the splits written by `Wik_eval.ipynb`.
2. **Load** the base model and score it as it is, before fine-tuning.
3. **Choose** the number of epochs on the validation split.
4. **Train** the final model on the 90% pool and test it once.
5. **Cross-validate** on the same 10 folds as the WFST model.
6. **Compare** the result with the WFST model.

In [1]:
import os
import random
import sys
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration
from transformers.optimization import Adafactor
from transformers.utils import logging as hf_logging

sys.dont_write_bytecode = True  # no __pycache__ in ../utils
sys.path.append(os.path.abspath("../utils"))
from tsv_metrics import tokenize_ipa, compute_per_wer

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

SMOKE_TEST = os.environ.get("G2P_SMOKE_TEST") == "1"

current_dir = Path().resolve()
dicts_dir   = current_dir / "dicts"
# A smoke test writes nothing into the project.
output_dir  = Path(tempfile.mkdtemp(prefix="byt5ft_smoke_")) if SMOKE_TEST else current_dir / "output"
model_dir   = output_dir / "byt5ft_model" if SMOKE_TEST else current_dir / "train" / "byt5ft_model"
output_dir.mkdir(exist_ok=True)

SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu" and not SMOKE_TEST:
    raise RuntimeError("PyTorch can't see a GPU. Install the CUDA build of PyTorch (see the setup "
                       "steps above); fine-tuning on a CPU would take days.")
print("Device:", torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
      "| smoke test:", SMOKE_TEST, "| outputs:", output_dir)

e:\College\Thesis\Projects\Taglog-G2P\.venv-windows\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: NVIDIA GeForce RTX 3060 | smoke test: False | outputs: E:\College\Thesis\Projects\Taglog-G2P\notebook\output


## 1. Load the splits

The dictionaries written by `Wik_eval.ipynb`, one word per line with its phones separated by
spaces. The model writes each transcription as one string without spaces (`ʔaɾaw`), the CharsiuG2P
format, and `tokenize_ipa()` splits it back into phones. An `assert` checks that this round trip
gives back exactly the reference phones for every entry, so the format itself costs no errors.

In [2]:
def read_dict(path):
    """Read a word TAB phones file into a list of (word, [phones])."""
    entries = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            word, phones = line.rstrip("\n").split("\t")
            entries.append((word, phones.split()))
    return entries


train    = read_dict(dicts_dir / "cwik_train.dict")
val      = read_dict(dicts_dir / "cwik_val.dict")
trainval = read_dict(dicts_dir / "cwik_trainval.dict")
test     = read_dict(dicts_dir / "cwik_test.tsv")
ref      = {w: p for w, p in test}     # a word listed twice keeps its last transcription
val_ref  = {w: p for w, p in val}

bad = [w for w, p in trainval + test if tokenize_ipa("".join(p)) != p]
assert not bad, f"phones don't survive the round trip: {bad[:5]}"

if SMOKE_TEST:
    train, val, trainval = train[:64], val[:32], trainval[:96]
    ref = dict(list(ref.items())[:32])
    val_ref = {w: p for w, p in val}

print(f"train {len(train)} | val {len(val)} | trainval {len(trainval)} | test words {len(ref)}")

train 17368 | val 2171 | trainval 19539 | test words 2171


## 2. Load the base model

The model and tokenizer are pinned to fixed Hugging Face revisions. CharsiuG2P takes the word after
a language tag, `<tgl>: ` for Tagalog, and its tokenizer is ByT5's (`google/byt5-small`), as in
the CharsiuG2P instructions. Words are decoded greedily (one beam), which the CharsiuG2P authors
found as good as beam search.

The base model is scored on the first 300 validation words before any fine-tuning, to show what
fine-tuning adds.

In [3]:
MODEL_ID       = "charsiu/g2p_multilingual_byT5_small_100"
MODEL_REVISION = "ed5cf99707fb39f5f3ccc461530d56b6fb531fd3"   # Hub commit of 2022-08-27
TOKENIZER_ID   = "google/byt5-small"
TOKENIZER_REV  = "68377bdc18a2ffec8a0533fef03b1c513a4dd49d"   # Hub commit of 2023-01-24
PREFIX         = "<tgl>: "
MAX_NEW_TOKENS = 128

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, revision=TOKENIZER_REV)


def load_base():
    """A fresh copy of the pretrained model."""
    return T5ForConditionalGeneration.from_pretrained(MODEL_ID, revision=MODEL_REVISION).to(device)


def predict(model, words, batch_size=128):
    """Return {word: [phones]}, decoded greedily."""
    model.eval()
    preds = {}
    for i in range(0, len(words), batch_size):
        batch = words[i:i + batch_size]
        enc = tokenizer([PREFIX + w for w in batch], padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(**enc, num_beams=1, max_new_tokens=MAX_NEW_TOKENS)
        for w, s in zip(batch, tokenizer.batch_decode(out, skip_special_tokens=True)):
            preds[w] = tokenize_ipa("".join(s.split()))
    return preds


base = load_base()
probe = list(val_ref)[:300]
m = compute_per_wer({w: val_ref[w] for w in probe}, predict(base, probe))
print(f"Base model on {len(probe)} validation words: PER {m['per'] * 100:.2f}%  WER {m['wer'] * 100:.2f}%")
for w in probe[:5]:
    print(f"  {w:<14} base:[{' '.join(predict(base, [w])[w])}]  ref:[{' '.join(val_ref[w])}]")
del base

e:\College\Thesis\Projects\Taglog-G2P\.venv-windows\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ikeep\.cache\huggingface\hub\models--google--byt5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
e:\College\Thesis\Projects\Taglog-G2P\.venv-windows\Lib\site-packages\huggingface_hub\file_download

Base model on 300 validation words: PER 35.27%  WER 98.67%
  satin          base:[s ɐ t i n]  ref:[s a t i n]
  mabilis        base:[m ɐ b ɪ l i s ɐ]  ref:[m a b i l i s]
  danaw          base:[d a n a ʊ̯]  ref:[d a n a w]
  anubing        base:[ʔ ɐ n ʊ b i ŋ]  ref:[ʔ a n u b i ŋ]
  minatayin      base:[m ɪ n ɐ t ɐ j i n]  ref:[m i n a t a j i n]


## 3. Choose the number of epochs on the validation split

The model is fine-tuned on the training split, and after every epoch it predicts the whole
validation split. The epoch with the lowest validation PER is kept. Training stops early when the
PER hasn't improved for two epochs.

The training settings follow Marqueses et al. (2026), who fine-tuned the same base model: the
Adafactor optimizer with a constant learning rate of 3 × 10⁻⁴. Batches have 32 words. The seed is
fixed, so a rerun on the same hardware gives the same model.

In [4]:
# Training settings. Lower BATCH_SIZE if the GPU runs out of memory.
LEARNING_RATE = 3e-4
BATCH_SIZE    = 8 if SMOKE_TEST else 32
MAX_EPOCHS    = 1 if SMOKE_TEST else 15
PATIENCE      = 2


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def fine_tune(entries, epochs, val_ref=None):
    """Fine-tune a fresh base model on (word, phones) entries.

    With val_ref, the validation PER is computed after every epoch, the best epoch's weights are
    kept and training stops after PATIENCE epochs without improvement. Returns the model and a list
    of per-epoch records.
    """
    set_seed()
    model = load_base()
    optimizer = Adafactor(model.parameters(), lr=LEARNING_RATE, relative_step=False,
                          scale_parameter=False, warmup_init=False)
    order = list(range(len(entries)))
    log, best_per, best_state, since_best = [], None, None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        random.shuffle(order)
        start, total_loss = time.time(), 0.0
        for i in range(0, len(order), BATCH_SIZE):
            batch = [entries[j] for j in order[i:i + BATCH_SIZE]]
            enc = tokenizer([PREFIX + w for w, _ in batch], padding=True, return_tensors="pt").to(device)
            labels = tokenizer(["".join(p) for _, p in batch], padding=True, return_tensors="pt").input_ids
            labels[labels == tokenizer.pad_token_id] = -100
            loss = model(**enc, labels=labels.to(device)).loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item() * len(batch)
        record = {"epoch": epoch, "train_loss": total_loss / len(entries), "seconds": round(time.time() - start)}

        if val_ref is not None:
            m = compute_per_wer(val_ref, predict(model, list(val_ref)))
            record.update(val_per=m["per"], val_wer=m["wer"])
            if best_per is None or m["per"] < best_per:
                best_per, since_best = m["per"], 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                since_best += 1
        log.append(record)
        print(record)
        if val_ref is not None and since_best >= PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, log


model, log = fine_tune(train, MAX_EPOCHS, val_ref)
log = pd.DataFrame(log)
log.to_csv(output_dir / "byt5ft_epochs.tsv", sep="\t", index=False)
BEST_EPOCHS = int(log.loc[log["val_per"].idxmin(), "epoch"])
del model
print(f"Best number of epochs: {BEST_EPOCHS}")
log

{'epoch': 1, 'train_loss': 0.032017570354888676, 'seconds': 224, 'val_per': 0.015058823529411765, 'val_wer': 0.07922616305849839}
{'epoch': 2, 'train_loss': 0.018152896492672935, 'seconds': 218, 'val_per': 0.013470588235294118, 'val_wer': 0.0695532012897282}
{'epoch': 3, 'train_loss': 0.012927451600768495, 'seconds': 238, 'val_per': 0.012647058823529412, 'val_wer': 0.06402579456471669}
{'epoch': 4, 'train_loss': 0.009987512831272662, 'seconds': 254, 'val_per': 0.012352941176470587, 'val_wer': 0.06172270842929528}
{'epoch': 5, 'train_loss': 0.007347590814576021, 'seconds': 266, 'val_per': 0.011352941176470588, 'val_wer': 0.06218332565637952}
{'epoch': 6, 'train_loss': 0.005697752649110526, 'seconds': 233, 'val_per': 0.009411764705882352, 'val_wer': 0.06540764624596962}
{'epoch': 7, 'train_loss': 0.00439658285209356, 'seconds': 232, 'val_per': 0.00988235294117647, 'val_wer': 0.06448641179180103}
{'epoch': 8, 'train_loss': 0.003217827186019122, 'seconds': 225, 'val_per': 0.012470588235294

,epoch,train_loss,seconds,val_per,val_wer
0,1,0.032018,224,0.015059,0.079226
1,2,0.018153,218,0.013471,0.069553
2,3,0.012927,238,0.012647,0.064026
3,4,0.009988,254,0.012353,0.061723
4,5,0.007348,266,0.011353,0.062183
5,6,0.005698,233,0.009412,0.065408
6,7,0.004397,232,0.009882,0.064486
7,8,0.003218,225,0.012471,0.061723


## 4. Train the final model and test it

The final model is trained for the chosen number of epochs on the 90% pool, the same data as the
final WFST model, and applied once to the test split. Its predictions are saved to
`output/byt5ft_output.txt` and the model to `train/byt5ft_model/` (1.2 GB, not tracked in git).

In [5]:
model, _ = fine_tune(trainval, BEST_EPOCHS)
model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)

hyp = predict(model, list(ref))
with open(output_dir / "byt5ft_output.txt", "w", encoding="utf-8") as f:
    for word, phones in hyp.items():
        f.write(f"{word}\t{' '.join(phones)}\n")
del model

metrics = compute_per_wer(ref, hyp)
assert metrics["n_words"] == len(ref), f"scored {metrics['n_words']} of {len(ref)} test words"
print("=== CharsiuG2P fine-tuned on the 90% pool, on the Wik_eval held-out test split ===")
print(f"  PER : {metrics['per']:.4f}  ({metrics['per'] * 100:.2f}%)   <- primary metric")
print(f"  WER : {metrics['wer']:.4f}  ({metrics['wer'] * 100:.2f}%)")
print(f"  Words evaluated  : {metrics['n_words']}")
print(f"  Correct words    : {metrics['n_correct']}")
print(f"  Total edit dist  : {metrics['n_edit']}")

{'epoch': 1, 'train_loss': 0.03167741059999759, 'seconds': 231}
{'epoch': 2, 'train_loss': 0.017275035502933883, 'seconds': 241}
{'epoch': 3, 'train_loss': 0.012342672099341841, 'seconds': 259}
{'epoch': 4, 'train_loss': 0.009872934002645717, 'seconds': 261}
{'epoch': 5, 'train_loss': 0.0066840119928244564, 'seconds': 225}
{'epoch': 6, 'train_loss': 0.005665683172999134, 'seconds': 217}
=== CharsiuG2P fine-tuned on the 90% pool, on the Wik_eval held-out test split ===
  PER : 0.0069  (0.69%)   <- primary metric
  WER : 0.0497  (4.97%)
  Words evaluated  : 2171
  Correct words    : 2063
  Total edit dist  : 119


## 5. Cross-validation

The same 10 folds as the cross-validation in section 5 of `Wik_eval.ipynb`, rebuilt from the 90%
pool with the same seed. Each round fine-tunes a fresh base model on nine folds for the chosen
number of epochs and tests it on the tenth, exactly as the WFST model is retrained there.

Each fold's PER is added to `output/byt5ft_cv.tsv` as soon as it is done. A rerun skips the folds
already in the file, so a stopped run can be resumed (delete the file to start over).

In [6]:
random.seed(42)
cv_entries = trainval[:]
random.shuffle(cv_entries)

N_FOLDS = 10
fold_size = len(cv_entries) // N_FOLDS
folds = [cv_entries[i * fold_size:(i + 1) * fold_size] for i in range(N_FOLDS)]
folds[-1] += cv_entries[N_FOLDS * fold_size:]   # remainder into last fold
print("Fold sizes:", [len(f) for f in folds])

cv_path = output_dir / "byt5ft_cv.tsv"
done = pd.read_csv(cv_path, sep="\t").to_dict("records") if cv_path.is_file() else []

for k in range(2 if SMOKE_TEST else N_FOLDS):
    if k + 1 in {r["fold"] for r in done}:
        continue
    train_fold = [e for i, f in enumerate(folds) if i != k for e in f]
    fold_ref = {w: p for w, p in folds[k]}
    model, _ = fine_tune(train_fold, BEST_EPOCHS)
    m = compute_per_wer(fold_ref, predict(model, list(fold_ref)))
    del model
    done.append({"fold": k + 1, "per": m["per"]})
    pd.DataFrame(done).to_csv(cv_path, sep="\t", index=False)
    print(f"  Fold {k + 1:2d}: PER {m['per'] * 100:5.2f}%  ({m['n_words']} words)")

fold_pers = pd.DataFrame(done).sort_values("fold")["per"].to_numpy(dtype=float)
print()
print(f"{len(fold_pers)}-fold CV PER: mean {fold_pers.mean() * 100:.2f}%  +/-  {fold_pers.std() * 100:.2f}%")

Fold sizes: [1953, 1953, 1953, 1953, 1953, 1953, 1953, 1953, 1953, 1962]
{'epoch': 1, 'train_loss': 0.032062511508133065, 'seconds': 198}
{'epoch': 2, 'train_loss': 0.018153102213937743, 'seconds': 197}
{'epoch': 3, 'train_loss': 0.01356270700606491, 'seconds': 196}
{'epoch': 4, 'train_loss': 0.010305273993871046, 'seconds': 197}
{'epoch': 5, 'train_loss': 0.00756248750116574, 'seconds': 193}
{'epoch': 6, 'train_loss': 0.005637055794489536, 'seconds': 195}
  Fold  1: PER  0.89%  (1953 words)
{'epoch': 1, 'train_loss': 0.03224295296774152, 'seconds': 208}
{'epoch': 2, 'train_loss': 0.01820712186731665, 'seconds': 203}
{'epoch': 3, 'train_loss': 0.013399642857812014, 'seconds': 196}


KeyboardInterrupt: 

## 6. Compare with the WFST model

The WFST predictions in `output/cwik_output.txt` are scored against the same reference.
`G2P_compare.ipynb` compares all the candidates, with significance tests.

In [7]:
def read_output(path):
    """Read a word TAB phones file into {word: [phones]}."""
    return {w: p for w, p in read_dict(path)}


rows = [{"Model": "CharsiuG2P fine-tuned", **metrics}]
cwik_output = current_dir / "output" / "cwik_output.txt"
if cwik_output.is_file():
    rows.insert(0, {"Model": "WFST (Wik_eval)", **compute_per_wer(ref, read_output(cwik_output))})

summary = pd.DataFrame(rows)
summary["PER (%)"] = (summary["per"] * 100).round(2)
summary["WER (%)"] = (summary["wer"] * 100).round(2)
summary[["Model", "PER (%)", "WER (%)", "n_words", "n_correct", "n_ref_phones", "n_edit"]]

,Model,PER (%),WER (%),n_words,n_correct,n_ref_phones,n_edit
0,WFST (Wik_eval),0.81,5.85,2171,2044,17303,141
1,CharsiuG2P fine-tuned,0.69,4.97,2171,2063,17303,119
